## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import os
from collections import defaultdict

import numpy as np

import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from scipy.sparse import issparse, diags
from scipy.stats import spearmanr, entropy as shannon_entropy
from statsmodels.stats.multitest import multipletests

# Load data


In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"
STOREPATH = os.path.join(BASEPATH, "correlation_analyses_proportions_direct")
os.makedirs(STOREPATH, exist_ok=True)

In [ ]:
# Load the proportions-based Harmony-integrated lineage file.
# .X contains z-scored proportions; .raw.X contains the original proportions [0, 1].
# The connectivity graph was built from proportions PCA + Harmony.
adata = sc.read_h5ad(
    os.path.join(
        BASEPATH,
        "UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony_proportions_lineage.h5ad",
    )
)

adata

# Check UMAP


In [ ]:
sc.pl.umap(
    adata,
    color=[
        "leiden",
        "phase",
        "batch_number",
        "dpt_pseudotime",
        "n_genes_by_counts",
        "celltype",
    ],
    wspace=0.4,
    ncols=2,
)

# Find genes changing upon pseudotime


In [ ]:
adata.obs.dpt_pseudotime.value_counts(dropna=False)

In [ ]:
display(adata)
display(adata[~adata.obs["dpt_pseudotime"].isna()])

In [ ]:
adata = adata[~adata.obs["dpt_pseudotime"].isna()].copy()
adata

In [ ]:
# Extract the raw (pre-scaling) proportions from .raw.X
# These are the within-gene UTR usage proportions in [0, 1].
raw_X = adata.raw.X
if issparse(raw_X):
    X_prop_raw = raw_X.toarray().astype(np.float64)
else:
    X_prop_raw = np.asarray(raw_X, dtype=np.float64)

print(f"Raw proportions matrix: {X_prop_raw.shape}")
print(f"Value range: [{X_prop_raw.min():.4f}, {X_prop_raw.max():.4f}]")
print(f"Sparsity (exact zeros): {(X_prop_raw == 0).mean():.1%}")

# Strategy: Identify genes with shifting 3'UTR isoform proportions along pseudotime

## Motivation

Each gene in this dataset can have multiple 3'UTR isoforms (alternative polyadenylation sites). The total expression of a gene must be _distributed_ across these sites, making this a **compositional / proportions** problem. We want to identify genes whose **relative usage of UTR isoforms** changes along the SVZ neurogenesis trajectory (`dpt_pseudotime`), indicating alternative polyadenylation (APA) regulation during differentiation.

## Approach: smooth proportions directly

This notebook computes within-gene UTR usage proportions from raw counts **before** batch integration, then applies Harmony on those proportions. The resulting proportions-based Harmony connectivity graph is used to smooth the proportions directly:

1. **Start with pre-computed proportions** from the proportions-based Harmony integration. The `.raw.X` in the loaded file contains the original proportions in [0, 1] before z-scoring.

2. **Graph-smooth the proportions** using the proportions-based Harmony-corrected neighbors graph (2 rounds of neighbor-averaging). Since proportions are compositional (within-gene sums = 1), smoothing preserves this property approximately, and we re-normalize afterward.

3. **Re-normalize** smoothed proportions within each gene so they sum to 1 per cell.

4. **Per-site Spearman ρ of proportion vs. pseudotime**: For each UTR site within multi-site genes, correlate its smoothed proportion with pseudotime.

5. **Per-gene Shannon entropy of UTR usage vs. pseudotime**: Correlate the entropy of the UTR proportion vector with pseudotime.

6. **Combine and rank** genes by both proportion-shift and entropy statistics.

## Comparison with the companion notebook

The companion notebook (`260309-genelevel-utr_pseudotime_spearman_graphsmoothed_proportions.ipynb`) uses a different approach: it smooths **raw counts** over the **counts-based** Harmony connectivity graph, then converts smoothed counts to proportions. This notebook instead smooths **proportions directly** over the **proportions-based** Harmony connectivity graph. Concordant hits between the two approaches are high-confidence candidates.

## Note on p-values

P-values in this analysis are **anti-conservative (inflated)** due to graph smoothing inducing spatial autocorrelation (effective N \u226a actual N) and the neighbors graph being correlated with pseudotime. We report p-values for completeness, but **gene selection relies on effect-size thresholds** (|ρ|) rather than p-value cutoffs.


In [ ]:
# Verify neighbors graph
print("Neighbors params:", adata.uns.get("neighbors", {}).get("params", {}))
print("Connectivities shape:", adata.obsp["connectivities"].shape)
print(f"Number of cells: {adata.n_obs}")
print(f"Number of UTR sites (features): {adata.n_vars}")

print(f"\nadata.var columns: {list(adata.var.columns)}")
display(adata.var.head(10))

## Build UTR site grouping by gene

We use the `name` column in `adata.var` (ENSEMBL gene ID) to group UTR sites belonging to the same gene. Only genes with \u22652 UTR sites are retained.


In [ ]:
# Build gene -> UTR site mapping
var_names = list(adata.var_names)
var_name_to_idx = {v: i for i, v in enumerate(var_names)}

gene_to_sites = defaultdict(list)
for site_name, row in adata.var.iterrows():
    gene_id = row["name"]  # ENSEMBL gene ID
    gene_to_sites[gene_id].append(site_name)

# Filter to multi-site genes only
multi_site_genes = {g: sites for g, sites in gene_to_sites.items() if len(sites) >= 2}

# Build gene_name lookup (ENSEMBL -> symbol)
gene_id_to_name = adata.var.groupby("name")["gene_name"].first().to_dict()

# Summary
n_sites_per_gene = [len(s) for s in multi_site_genes.values()]
print(f"Total genes: {len(gene_to_sites)}")
print(f"Multi-site genes (\u22652 UTR sites): {len(multi_site_genes)}")
print(f"Single-site genes (excluded): {len(gene_to_sites) - len(multi_site_genes)}")
print(f"Total UTR sites in multi-site genes: {sum(n_sites_per_gene)}")
print(f"\nDistribution of sites per gene (multi-site only):")
sites_series = pd.Series(n_sites_per_gene)
print(sites_series.describe())
print(f"\nSites per gene value counts:")
print(sites_series.value_counts().sort_index())

## Graph-smooth proportions

We row-normalize the proportions-based Harmony-corrected connectivities matrix and apply 2 rounds of neighbor-averaging to denoise the proportions while implicitly correcting batch effects through the graph topology.

Since each cell's proportions sum to 1 within each gene, and smoothing computes weighted averages over neighbors (with weights summing to 1), the smoothed proportions approximately preserve the compositional constraint. We re-normalize afterward to ensure exact sums of 1.


In [ ]:
# Row-normalize the Harmony-corrected connectivities matrix
conn = adata.obsp["connectivities"].copy()

row_sums = np.array(conn.sum(axis=1)).flatten()
row_sums[row_sums == 0] = 1  # avoid division by zero for isolated cells
norm_conn = diags(1.0 / row_sums) @ conn

# Apply iterative graph smoothing to RAW PROPORTIONS
n_rounds = 2
X_smooth = X_prop_raw.copy()
for i in range(n_rounds):
    X_smooth = norm_conn @ X_smooth
    print(f"Smoothing round {i + 1}/{n_rounds} complete")

print(f"\nSmoothed proportions matrix shape: {X_smooth.shape}")
print(f"Sparsity before smoothing: {(X_prop_raw == 0).mean():.1%}")
print(f"Sparsity after smoothing:  {(X_smooth == 0).mean():.1%}")

## Re-normalize smoothed proportions

After graph smoothing, within-gene proportions may not sum to exactly 1 (e.g. if some neighbors had zero expression for that gene). Re-normalize so each gene's UTR site proportions sum to 1 per cell. Cells with zero total for a gene get uniform proportions as a conservative default.


In [ ]:
# Re-normalize smoothed proportions within each gene
P_smooth = np.zeros_like(X_smooth)

for gene_id, site_names in multi_site_genes.items():
    site_idxs = [var_name_to_idx[s] for s in site_names]
    # Sum across this gene's UTR sites per cell
    gene_total = X_smooth[:, site_idxs].sum(axis=1, keepdims=True)
    # Avoid division by zero: where gene total is 0, set uniform proportions
    zero_mask = gene_total.flatten() == 0
    n_sites = len(site_idxs)
    for j, idx in enumerate(site_idxs):
        P_smooth[:, idx] = np.where(
            zero_mask,
            1.0 / n_sites,
            X_smooth[:, idx] / gene_total.flatten(),
        )

# Sanity check: proportions sum to 1 within each gene
sample_gene = list(multi_site_genes.keys())[0]
sample_sites = multi_site_genes[sample_gene]
sample_idxs = [var_name_to_idx[s] for s in sample_sites]
prop_sums = P_smooth[:, sample_idxs].sum(axis=1)
print(
    f"Sanity check \u2014 gene {sample_gene} ({gene_id_to_name.get(sample_gene, '?')}):"
)
print(f"  Sites: {sample_sites}")
print(
    f"  Proportion sums: min={prop_sums.min():.6f}, max={prop_sums.max():.6f}, mean={prop_sums.mean():.6f}"
)
print(f"\nProportions matrix shape: {P_smooth.shape}")

## Per-site Spearman ρ of proportion vs. pseudotime

For each UTR site within multi-site genes, we compute the Spearman correlation between the site's smoothed **proportion** and pseudotime.


In [ ]:
pseudotime = adata.obs["dpt_pseudotime"].values

# Collect all UTR sites from multi-site genes
all_multi_sites = [
    (gene_id, site) for gene_id, sites in multi_site_genes.items() for site in sites
]

results_site = []
for i, (gene_id, site_name) in enumerate(all_multi_sites):
    if (i + 1) % 2000 == 0 or i == 0:
        print(f"Processing site {i + 1}/{len(all_multi_sites)}...")
    idx = var_name_to_idx[site_name]
    prop = P_smooth[:, idx]
    if np.std(prop) == 0:
        results_site.append(
            (gene_id, gene_id_to_name.get(gene_id, ""), site_name, 0.0, 1.0)
        )
        continue
    rho, pval = spearmanr(pseudotime, prop)
    results_site.append(
        (gene_id, gene_id_to_name.get(gene_id, ""), site_name, rho, pval)
    )

print(
    f"Done. Tested {len(results_site)} UTR sites across {len(multi_site_genes)} genes."
)

df_site = pd.DataFrame(
    results_site, columns=["gene_id", "gene_name", "site", "spearman_rho", "pvalue"]
)

# BH FDR correction across all sites
_, df_site["padj"], _, _ = multipletests(df_site["pvalue"], method="fdr_bh")
df_site["abs_rho"] = df_site["spearman_rho"].abs()
df_site = df_site.sort_values("abs_rho", ascending=False).reset_index(drop=True)

print(f"\n--- Per-site summary ---")
print(f"Total sites tested: {len(df_site)}")
print(f"Significant (padj < 0.05): {(df_site['padj'] < 0.05).sum()}")
print(f"|\u03c1| > 0.3: {(df_site['abs_rho'] > 0.3).sum()}")
print(
    f"|\u03c1| > 0.3 & padj < 0.05: {((df_site['abs_rho'] > 0.3) & (df_site['padj'] < 0.05)).sum()}"
)

df_site.head(20)

## Gene-level summary of proportion shifts

For each gene, we summarize APA dynamics by: (1) the site with the maximum |ρ|, indicating the strongest directional shift; (2) the number of significant sites.


In [ ]:
# Gene-level summary: for each gene, get the site with max |rho|
gene_summary_rows = []
for gene_id, group in df_site.groupby("gene_id"):
    best_row = group.loc[group["abs_rho"].idxmax()]
    n_sig = ((group["abs_rho"] > 0.3) & (group["padj"] < 0.05)).sum()
    gene_summary_rows.append(
        {
            "gene_id": gene_id,
            "gene_name": best_row["gene_name"],
            "n_sites": len(group),
            "best_site": best_row["site"],
            "best_rho": best_row["spearman_rho"],
            "best_abs_rho": best_row["abs_rho"],
            "best_padj": best_row["padj"],
            "n_sig_sites": n_sig,
        }
    )

df_gene_prop = (
    pd.DataFrame(gene_summary_rows)
    .sort_values("best_abs_rho", ascending=False)
    .reset_index(drop=True)
)

print(f"--- Gene-level proportion-shift summary ---")
print(f"Total multi-site genes: {len(df_gene_prop)}")
print(
    f"Genes with |\u03c1| > 0.3 (best site): {(df_gene_prop['best_abs_rho'] > 0.3).sum()}"
)
print(
    f"Genes with |\u03c1| > 0.2 (best site): {(df_gene_prop['best_abs_rho'] > 0.2).sum()}"
)
print()

display(
    df_gene_prop.head(20).style.format(
        {"best_rho": "{:.3f}", "best_abs_rho": "{:.3f}", "best_padj": "{:.2e}"}
    )
)

## Shannon entropy of UTR usage vs. pseudotime

Shannon entropy measures how _spread out_ a gene's expression is across its UTR sites:

- **High entropy** -> the gene uses multiple UTR sites roughly equally (diverse/uniform usage)
- **Low entropy** -> one UTR site dominates (deterministic usage)

A negative correlation of entropy with pseudotime means the gene's UTR usage becomes **more deterministic** as differentiation progresses. A positive correlation means the gene **diversifies** its UTR usage.


In [ ]:
# Compute per-cell per-gene Shannon entropy of UTR proportions (vectorized)


def _shannon_entropy_rows(p):
    """Vectorized Shannon entropy (base 2) for each row of matrix p.
    Handles 0*log(0)=0 correctly."""
    with np.errstate(divide="ignore", invalid="ignore"):
        log_p = np.where(p > 0, np.log2(p), 0.0)
    return -np.sum(p * log_p, axis=1)


entropy_results = []
gene_ids_ordered = list(multi_site_genes.keys())
entropy_matrix = np.zeros((adata.n_obs, len(gene_ids_ordered)))

for g_idx, gene_id in enumerate(gene_ids_ordered):
    if (g_idx + 1) % 1000 == 0 or g_idx == 0:
        print(f"Computing entropy for gene {g_idx + 1}/{len(gene_ids_ordered)}...")
    site_names = multi_site_genes[gene_id]
    site_idxs = [var_name_to_idx[s] for s in site_names]
    # Get proportions for this gene's sites (cells x sites)
    props = P_smooth[:, site_idxs]
    # Vectorized Shannon entropy — all cells at once instead of per-cell loop
    H = _shannon_entropy_rows(props)
    k = len(site_names)
    H = H / np.log2(k)  # normalize to [0, 1]
    entropy_matrix[:, g_idx] = H

    # Correlate entropy with pseudotime
    if np.std(H) == 0:
        entropy_results.append(
            (
                gene_id,
                gene_id_to_name.get(gene_id, ""),
                len(site_names),
                0.0,
                1.0,
                np.mean(H),
            )
        )
    else:
        rho, pval = spearmanr(pseudotime, H)
        entropy_results.append(
            (
                gene_id,
                gene_id_to_name.get(gene_id, ""),
                len(site_names),
                rho,
                pval,
                np.mean(H),
            )
        )

print(f"Done. Computed entropy for {len(entropy_results)} genes.")

df_entropy = pd.DataFrame(
    entropy_results,
    columns=[
        "gene_id",
        "gene_name",
        "n_sites",
        "entropy_rho",
        "entropy_pval",
        "mean_norm_entropy",
    ],
)

# BH FDR correction
_, df_entropy["entropy_padj"], _, _ = multipletests(
    df_entropy["entropy_pval"], method="fdr_bh"
)
df_entropy["entropy_abs_rho"] = df_entropy["entropy_rho"].abs()
df_entropy = df_entropy.sort_values("entropy_abs_rho", ascending=False).reset_index(
    drop=True
)

print(f"\n--- Entropy analysis summary ---")
print(f"Genes tested: {len(df_entropy)}")
print(f"|ρ| > 0.3: {(df_entropy['entropy_abs_rho'] > 0.3).sum()}")
print(f"|ρ| > 0.2: {(df_entropy['entropy_abs_rho'] > 0.2).sum()}")
print(
    f"\nEntropy decreasing (more deterministic with pseudotime, ρ < -0.2): {(df_entropy['entropy_rho'] < -0.2).sum()}"
)
print(
    f"Entropy increasing (more diverse with pseudotime, ρ > 0.2): {(df_entropy['entropy_rho'] > 0.2).sum()}"
)

df_entropy.head(20)

## Combined gene-level results

Merge the proportion-shift and entropy analyses into a single gene-level table. Genes significant in both analyses are the highest-confidence APA candidates.


In [ ]:
# Merge proportion-shift and entropy results
df_combined = df_gene_prop.merge(
    df_entropy[
        [
            "gene_id",
            "entropy_rho",
            "entropy_pval",
            "entropy_padj",
            "entropy_abs_rho",
            "mean_norm_entropy",
        ]
    ],
    on="gene_id",
    how="inner",
)

# Flag genes significant in both analyses
rho_thresh = 0.2
df_combined["prop_sig"] = df_combined["best_abs_rho"] > rho_thresh
df_combined["entr_sig"] = df_combined["entropy_abs_rho"] > rho_thresh
df_combined["both_sig"] = df_combined["prop_sig"] & df_combined["entr_sig"]

# Combined ranking score: geometric mean of the two |rho| values
df_combined["combined_score"] = np.sqrt(
    df_combined["best_abs_rho"] * df_combined["entropy_abs_rho"]
)
df_combined = df_combined.sort_values("combined_score", ascending=False).reset_index(
    drop=True
)

print(f"--- Combined analysis summary ---")
print(f"Total multi-site genes: {len(df_combined)}")
print(f"Proportion shift |ρ| > {rho_thresh}: {df_combined['prop_sig'].sum()}")
print(f"Entropy shift |ρ| > {rho_thresh}: {df_combined['entr_sig'].sum()}")
print(f"Both significant: {df_combined['both_sig'].sum()}")
print()
print("Top 20 genes by combined score:")
display(
    df_combined.head(20)[
        [
            "gene_id",
            "gene_name",
            "n_sites",
            "best_site",
            "best_rho",
            "entropy_rho",
            "combined_score",
            "both_sig",
        ]
    ].style.format(
        {"best_rho": "{:.3f}", "entropy_rho": "{:.3f}", "combined_score": "{:.3f}"}
    )
)

## Visualization: Proportion-shift ρ vs. entropy ρ scatter

Each point is a gene. The x-axis shows the _directional_ shift of the strongest site (signed ρ), while the y-axis shows whether the gene's overall UTR diversity increases or decreases along pseudotime.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

# Color by significance category
colors = np.full(len(df_combined), "lightgrey", dtype=object)
colors[df_combined["prop_sig"].values & ~df_combined["entr_sig"].values] = "orange"
colors[~df_combined["prop_sig"].values & df_combined["entr_sig"].values] = (
    "mediumseagreen"
)
colors[df_combined["both_sig"].values] = "firebrick"

ax.scatter(
    df_combined["best_rho"],
    df_combined["entropy_rho"],
    c=colors,
    s=8,
    alpha=0.6,
    edgecolors="none",
)

# Label top combined-score genes
n_label = 12
top_labeled = df_combined[df_combined["both_sig"]].head(n_label)
for _, row in top_labeled.iterrows():
    ax.annotate(
        row["gene_name"],
        (row["best_rho"], row["entropy_rho"]),
        fontsize=7,
        alpha=0.85,
        ha="left",
        va="bottom",
    )

ax.axhline(0, color="black", linewidth=0.3)
ax.axvline(0, color="black", linewidth=0.3)
ax.axhline(-rho_thresh, color="grey", linestyle="--", linewidth=0.5, alpha=0.5)
ax.axhline(rho_thresh, color="grey", linestyle="--", linewidth=0.5, alpha=0.5)
ax.axvline(-rho_thresh, color="grey", linestyle="--", linewidth=0.5, alpha=0.5)
ax.axvline(rho_thresh, color="grey", linestyle="--", linewidth=0.5, alpha=0.5)

ax.set_xlabel("Best site proportion \u03c1 (vs. pseudotime)", fontsize=12)
ax.set_ylabel("Entropy \u03c1 (vs. pseudotime)", fontsize=12)
ax.set_title(
    "UTR isoform dynamics: directional shift vs. diversification\n(graph-smoothed proportions direct, proportions-based Harmony)",
    fontsize=13,
)

legend_elements = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="firebrick",
        markersize=7,
        label=f"Both |\u03c1| > {rho_thresh}",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="orange",
        markersize=7,
        label=f"Proportion only |\u03c1| > {rho_thresh}",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="mediumseagreen",
        markersize=7,
        label=f"Entropy only |\u03c1| > {rho_thresh}",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="lightgrey",
        markersize=7,
        label="Not significant",
    ),
]
ax.legend(handles=legend_elements, loc="upper center", fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.show()

## Stacked area plots: UTR isoform proportions along pseudotime

The definitive visualization of APA dynamics. For each top-hit gene, we plot the mean proportion of each UTR site across binned pseudotime as a stacked area chart.


In [ ]:
def plot_stacked_proportions(
    gene_ids,
    df_combined,
    multi_site_genes,
    P_smooth,
    var_name_to_idx,
    pseudotime,
    gene_id_to_name,
    n_bins=40,
    ncols=3,
    adata_var=None,
):
    """Stacked area plots of UTR proportions along pseudotime for given genes."""
    n_genes = len(gene_ids)
    nrows = int(np.ceil(n_genes / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(5 * ncols, 3.5 * nrows), squeeze=False
    )

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_indices = np.digitize(pseudotime, bin_edges) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    cmap = plt.cm.Set2

    for g_idx, gene_id in enumerate(gene_ids):
        ax = axes[g_idx // ncols, g_idx % ncols]
        site_names = multi_site_genes[gene_id]
        site_idxs = [var_name_to_idx[s] for s in site_names]

        # Sort sites by UTR size if available
        if adata_var is not None and "utr_size" in adata_var.columns:
            site_sizes = [(s, adata_var.loc[s, "utr_size"]) for s in site_names]
            site_sizes.sort(key=lambda x: x[1])
            site_names_sorted = [s for s, _ in site_sizes]
            site_idxs_sorted = [var_name_to_idx[s] for s in site_names_sorted]
            labels = [
                f"{s.split('_')[-1] if '_' in s else s} ({int(sz)}nt)"
                for s, sz in site_sizes
            ]
        else:
            site_names_sorted = site_names
            site_idxs_sorted = site_idxs
            labels = site_names_sorted

        # Compute binned mean proportions per site
        binned_props = np.zeros((n_bins, len(site_idxs_sorted)))
        for b in range(n_bins):
            mask = bin_indices == b
            if mask.sum() > 0:
                for s_idx, feat_idx in enumerate(site_idxs_sorted):
                    binned_props[b, s_idx] = P_smooth[mask, feat_idx].mean()

        # Normalize bins to sum to 1 (safety)
        row_sums = binned_props.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        binned_props = binned_props / row_sums

        # Stacked area
        ax.stackplot(
            bin_centers,
            binned_props.T,
            labels=labels,
            colors=[cmap(i) for i in range(len(site_idxs_sorted))],
            alpha=0.85,
        )

        gene_name = gene_id_to_name.get(gene_id, gene_id)
        # Get rho from df_combined
        gene_row = df_combined[df_combined["gene_id"] == gene_id]
        if len(gene_row) > 0:
            best_rho = gene_row.iloc[0]["best_rho"]
            ent_rho = gene_row.iloc[0]["entropy_rho"]
            ax.set_title(
                f"{gene_name}\nprop \u03c1={best_rho:.2f}, entr \u03c1={ent_rho:.2f}",
                fontsize=9,
            )
        else:
            ax.set_title(gene_name, fontsize=9)

        ax.set_xlabel("Pseudotime", fontsize=8)
        ax.set_ylabel("UTR proportion", fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_xlim(0, 1)
        if len(labels) <= 5:
            ax.legend(fontsize=6, loc="upper right", framealpha=0.7)

    for idx in range(n_genes, nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    plt.tight_layout()
    plt.show()


# Plot top 12 genes by combined score
top_genes = df_combined[df_combined["both_sig"]].head(12)["gene_id"].tolist()
if top_genes:
    print(
        f"Top {len(top_genes)} genes significant in both proportion-shift and entropy:"
    )
    plot_stacked_proportions(
        top_genes,
        df_combined,
        multi_site_genes,
        P_smooth,
        var_name_to_idx,
        pseudotime,
        gene_id_to_name,
        ncols=3,
        adata_var=adata.var,
    )

## Stacked area plots: top proportion-shift-only genes

Genes with the strongest directional shift in a single UTR site, regardless of entropy change.


In [ ]:
# Top 12 genes by proportion shift only (strongest directional APA)
top_prop_genes = df_gene_prop.head(12)["gene_id"].tolist()
print(f"Top {len(top_prop_genes)} genes by strongest proportion shift:")
plot_stacked_proportions(
    top_prop_genes,
    df_combined,
    multi_site_genes,
    P_smooth,
    var_name_to_idx,
    pseudotime,
    gene_id_to_name,
    ncols=3,
    adata_var=adata.var,
)

## Entropy trends along pseudotime for top genes

Binned pseudotime showing mean \u00b1 SEM of Shannon entropy for the genes with the strongest entropy changes.


In [ ]:
def plot_entropy_along_pseudotime(
    gene_ids,
    entropy_matrix,
    gene_ids_ordered,
    pseudotime,
    gene_id_to_name,
    df_entropy,
    n_bins=40,
    ncols=3,
):
    """Plot binned entropy along pseudotime for given genes."""
    n_genes = len(gene_ids)
    nrows = int(np.ceil(n_genes / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows), squeeze=False
    )

    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_indices = np.digitize(pseudotime, bin_edges) - 1
    bin_indices = np.clip(bin_indices, 0, n_bins - 1)

    gene_idx_map = {g: i for i, g in enumerate(gene_ids_ordered)}
    ent_rho_lookup = df_entropy.set_index("gene_id")["entropy_rho"]

    for g_idx, gene_id in enumerate(gene_ids):
        ax = axes[g_idx // ncols, g_idx % ncols]
        col_idx = gene_idx_map[gene_id]
        H = entropy_matrix[:, col_idx]

        bin_means = np.array(
            [
                H[bin_indices == b].mean() if (bin_indices == b).sum() > 0 else np.nan
                for b in range(n_bins)
            ]
        )
        bin_sems = np.array(
            [
                (
                    H[bin_indices == b].std() / np.sqrt((bin_indices == b).sum())
                    if (bin_indices == b).sum() > 1
                    else 0
                )
                for b in range(n_bins)
            ]
        )

        rho = ent_rho_lookup.get(gene_id, 0)
        color = "steelblue" if rho < 0 else "firebrick"

        ax.fill_between(
            bin_centers,
            bin_means - bin_sems,
            bin_means + bin_sems,
            alpha=0.3,
            color=color,
        )
        ax.plot(bin_centers, bin_means, color=color, linewidth=1.5)
        gene_name = gene_id_to_name.get(gene_id, gene_id)
        ax.set_title(f"{gene_name}\nentropy ρ = {rho:.3f}", fontsize=10)
        ax.set_xlabel("Pseudotime")
        ax.set_ylabel("Normalized Shannon entropy")

    for idx in range(n_genes, nrows * ncols):
        axes[idx // ncols, idx % ncols].set_visible(False)

    plt.tight_layout()
    plt.show()


# Top 9 genes with decreasing entropy (becoming more deterministic)
top_decreasing_ent = (
    df_entropy[df_entropy["entropy_rho"] < 0].head(9)["gene_id"].tolist()
)
if top_decreasing_ent:
    print("Genes becoming more deterministic in UTR usage (entropy decreasing):")
    plot_entropy_along_pseudotime(
        top_decreasing_ent,
        entropy_matrix,
        gene_ids_ordered,
        pseudotime,
        gene_id_to_name,
        df_entropy,
        ncols=3,
    )

# Top 9 genes with increasing entropy (diversifying)
top_increasing_ent = (
    df_entropy[df_entropy["entropy_rho"] > 0].head(9)["gene_id"].tolist()
)
if top_increasing_ent:
    print("\nGenes diversifying UTR usage (entropy increasing):")
    plot_entropy_along_pseudotime(
        top_increasing_ent,
        entropy_matrix,
        gene_ids_ordered,
        pseudotime,
        gene_id_to_name,
        df_entropy,
        ncols=3,
    )

## Distribution of ρ values

Overview of the per-site proportion ρ values and per-gene entropy ρ values across all tested features.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Per-site proportion rho distribution
axes[0].hist(
    df_site["spearman_rho"], bins=80, color="slategrey", edgecolor="none", alpha=0.8
)
axes[0].axvline(0, color="black", linewidth=0.5)
axes[0].axvline(
    0.3, color="firebrick", linestyle="--", linewidth=1, label="|\u03c1| = 0.3"
)
axes[0].axvline(-0.3, color="firebrick", linestyle="--", linewidth=1)
axes[0].set_xlabel("Spearman \u03c1 (proportion vs. pseudotime)")
axes[0].set_ylabel("Number of UTR sites")
axes[0].set_title("Per-site proportion correlations")
axes[0].legend(fontsize=8)

# Per-gene max proportion rho distribution
axes[1].hist(
    df_gene_prop["best_rho"], bins=60, color="darkorange", edgecolor="none", alpha=0.8
)
axes[1].axvline(0, color="black", linewidth=0.5)
axes[1].axvline(
    0.2, color="firebrick", linestyle="--", linewidth=1, label="|\u03c1| = 0.2"
)
axes[1].axvline(-0.2, color="firebrick", linestyle="--", linewidth=1)
axes[1].set_xlabel("Best site \u03c1 (proportion vs. pseudotime)")
axes[1].set_ylabel("Number of genes")
axes[1].set_title("Gene-level proportion shift")
axes[1].legend(fontsize=8)

# Per-gene entropy rho distribution
axes[2].hist(
    df_entropy["entropy_rho"],
    bins=60,
    color="mediumseagreen",
    edgecolor="none",
    alpha=0.8,
)
axes[2].axvline(0, color="black", linewidth=0.5)
axes[2].axvline(
    0.2, color="firebrick", linestyle="--", linewidth=1, label="|\u03c1| = 0.2"
)
axes[2].axvline(-0.2, color="firebrick", linestyle="--", linewidth=1)
axes[2].set_xlabel("Entropy \u03c1 (vs. pseudotime)")
axes[2].set_ylabel("Number of genes")
axes[2].set_title("Gene-level entropy shift")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Save results


In [ ]:
# Save per-site proportion correlation results
outpath_site = os.path.join(
    STOREPATH, "utr_pseudotime_spearman_proportions_per_site.csv"
)
df_site.to_csv(outpath_site, index=False)
print(f"Per-site results saved to: {outpath_site}")

# Save gene-level proportion-shift results
outpath_gene_prop = os.path.join(
    STOREPATH, "utr_pseudotime_spearman_proportions_per_gene.csv"
)
df_gene_prop.to_csv(outpath_gene_prop, index=False)
print(f"Gene-level proportion results saved to: {outpath_gene_prop}")

# Save gene-level entropy results
outpath_entropy = os.path.join(STOREPATH, "utr_pseudotime_entropy_per_gene.csv")
df_entropy.to_csv(outpath_entropy, index=False)
print(f"Entropy results saved to: {outpath_entropy}")

# Save combined gene-level results
outpath_combined = os.path.join(
    STOREPATH, "utr_pseudotime_proportions_entropy_combined.csv"
)
df_combined.to_csv(outpath_combined, index=False)
print(f"Combined results saved to: {outpath_combined}")

# Store proportions as a layer in adata
adata.layers["smoothed_proportions"] = P_smooth

# Save adata with new layers
outpath_adata = os.path.join(
    BASEPATH, "UTR_count_table_genelevel_harmony_proportions_lineage_smoothed.h5ad"
)
adata.write_h5ad(outpath_adata)
print(f"AnnData saved to: {outpath_adata}")

print(f"\n--- Final summary ---")
print(f"Cells: {adata.n_obs}")
print(f"Multi-site genes analyzed: {len(df_combined)}")
print(
    f"Genes with proportion shift |\u03c1| > {rho_thresh}: {df_combined['prop_sig'].sum()}"
)
print(
    f"Genes with entropy shift |\u03c1| > {rho_thresh}: {df_combined['entr_sig'].sum()}"
)
print(f"Genes significant in both: {df_combined['both_sig'].sum()}")